# StateLM — separate retraining and playground cells

Your completed v4 run stays in Drive. This notebook adds a new training run and a real-model playground in separate cells.

1. Run **Set up and preserve current results**.
2. Run **Retrain recovery** when ready. This starts five fresh adapters and saves them to a new folder.
3. Run **Playground** and click **Load model**. You can select either the original saved run or the newly trained run. To try the original model immediately, skip retraining.

All source code and data are included. **Original data remains original; synthetic Test N is off by default.**
The revised results are development comparisons because we have already inspected the first run on these participants.


## 1. Set up and preserve current results
Run this once. It backs up the original result JSON files and leaves the original model adapters in their existing folder.


In [ ]:
from google.colab import drive
drive.flush_and_unmount(); drive.mount('/content/drive')
import os
base='/content/drive/MyDrive/StateLM_v4_1_runs'
print('in runs root:', os.listdir(base))
run=base+'/20260913_110344_527137'
print('run exists:', os.path.exists(run))
if os.path.exists(run):
    print('contents:', os.listdir(run))
    print('config exists:', os.path.exists(run+'/config.json'))
    print('seed_42 exists:', os.path.exists(run+'/seed_42'))

In [ ]:
import base64, hashlib, io, json, os, pathlib, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
PACKAGE_BYTES = base64.b64decode('...')
PACKAGE_SHA = '3a826262236a2596529f810ced1beaf8f25778121891e20535fc530f5541c9d2'
assert hashlib.sha256(PACKAGE_BYTES).hexdigest() == PACKAGE_SHA
V41_PACKAGE_DIR = pathlib.Path('/content') / ('statelm_v4_1_' + PACKAGE_SHA[:12])
V41_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(PACKAGE_BYTES)) as archive:
    for member in archive.infolist():
        if V41_PACKAGE_DIR.resolve() not in (V41_PACKAGE_DIR/member.filename).resolve().parents:
            raise ValueError('Invalid package path')
    archive.extractall(V41_PACKAGE_DIR)
for name, expected in json.loads((V41_PACKAGE_DIR/'MANIFEST.json').read_text()).items():
    assert hashlib.sha256((V41_PACKAGE_DIR/name).read_bytes()).hexdigest() == expected, name

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Choose a GPU runtime, then run this cell again.')
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],check=True)
V41_PY = globals().get('RUNTIME_PY')
subprocess.run([sys.executable,'-m','pip','install','-q','uv==0.6.17'],check=True)
uv=shutil.which('uv')
if not V41_PY or not pathlib.Path(V41_PY).exists():
    subprocess.run([uv,'python','install','3.12.10'],check=True)
    runtime=V41_PACKAGE_DIR/'.runtime'
    if not (runtime/'bin/python').exists():
        subprocess.run([uv,'venv','--python','3.12.10',str(runtime)],check=True)
    V41_PY=str(runtime/'bin/python')
subprocess.run([uv,'pip','install','--python',V41_PY,'-r',str(V41_PACKAGE_DIR/'requirements-colab.txt')],check=True)
os.environ['HF_HOME']='/content/statelm_v4_hf_cache'
os.environ['TOKENIZERS_PARALLELISM']='false'
if not os.environ.get('HF_TOKEN'):
    from google.colab import userdata
    try:os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')
    except Exception:
        import getpass
        os.environ['HF_TOKEN']=getpass.getpass('Hugging Face token with Gemma access: ')
from google.colab import drive
if not pathlib.Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ORIGINAL_RUN_DIR=pathlib.Path('/content/drive/MyDrive/StateLM_v4_1_runs/20260913_110344_527137')
V41_RESULTS_ROOT=pathlib.Path('/content/drive/MyDrive/StateLM_v4_1_runs')
V41_RESULTS_ROOT.mkdir(parents=True,exist_ok=True)
stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
if not ORIGINAL_RUN_DIR.exists():raise FileNotFoundError('Original saved run not found. Check that the correct Google Drive account is mounted.')
backup=V41_RESULTS_ROOT/('saved_original_results_'+stamp+'.zip')
hashes={}
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ORIGINAL_RUN_DIR.rglob('*.json')):
        name=str(path.relative_to(ORIGINAL_RUN_DIR));data=path.read_bytes()
        archive.writestr(name,data);hashes[name]=hashlib.sha256(data).hexdigest()
    archive.writestr('BACKUP_MANIFEST.json',json.dumps({'original_run':str(ORIGINAL_RUN_DIR),'sha256':hashes},indent=2))
with zipfile.ZipFile(backup) as archive:
    for name, expected in hashes.items():assert hashlib.sha256(archive.read(name)).hexdigest()==expected
print('Saved and verified the current results:',backup)
print('Original adapters remain at:',ORIGINAL_RUN_DIR)
check=subprocess.run([V41_PY,'-B','-m','unittest','discover','-s','tests'],cwd=V41_PACKAGE_DIR,capture_output=True,text=True)
if check.returncode:
    print(check.stdout,check.stderr)
    raise RuntimeError('Software checks failed. Training has not started.')
print('Software checks passed. No benchmark training has run in this setup cell.')
print('You can now retrain below, or skip to the playground to try your original saved models.')


## 2. Retrain recovery — a separate cell

This creates a **new run**, using all five folds and original survey preferences. It never loads your old trained adapters.
Training rewards choosing any recorded preference, discourages unnecessary changes to the base model, and stops when validation agreement stops improving.
The base model remains a validation candidate. The report shows the trained model separately even if validation selects the base.

`RUN_TEST_N = False` means the optional synthetic test is **off**. It does not disable real training.


In [ ]:
RUN_TEST_N = False
FOLDS = [0,1,2,3,4]
TRAINING_SEEDS = [42]
if '_statelm_tester' in globals():_statelm_tester.stop()
V41_RUN_DIR=V41_RESULTS_ROOT/datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
print('Starting fresh training on original data. Folds:',FOLDS)
print('Synthetic Test N:', 'separate test enabled' if RUN_TEST_N else 'off')
print('New results folder:',V41_RUN_DIR)
command=[V41_PY,'-B','-u','-m','statelm','train','--output',str(V41_RUN_DIR),'--folds',*map(str,FOLDS),'--seeds',*map(str,TRAINING_SEEDS)]
if RUN_TEST_N:command.append('--test-n')
log_path=V41_RESULTS_ROOT/(V41_RUN_DIR.name+'_console.log')
with log_path.open('w') as log:
    process=subprocess.Popen(command,cwd=V41_PACKAGE_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        result=process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:process.wait(timeout=15)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
if result:raise RuntimeError('Training stopped with an error. The console log and completed folds are saved: '+str(log_path))
print('Training and evaluation completed. Run the playground cell below to try the new models.')


## 3. Read the new results
All original results and optional Test N results remain separate. The original 62.4% base / 50.5% trained recovery result stays saved.


In [ ]:
from IPython.display import display, Markdown
if 'V41_RUN_DIR' not in globals():
    print('No new training run in this session. You can still use the playground with the original saved run.')
else:
    for seed in TRAINING_SEEDS:
        for suite in (['original','test_n'] if RUN_TEST_N else ['original']):
            path=V41_RUN_DIR/f'seed_{seed}'/f'{suite}_summary.json'
            if not path.exists():continue
            report=json.loads(path.read_text())
            rows=[f'### New {suite} results — seed {seed}','','| System | Task | Cases | Accuracy / preference agreement | Most-used recovery strategy | Median / p95 seconds |','|---|---|---:|---:|---:|---:|']
            for name,tasks in report['conditions'].items():
                for task,m in tasks.items():
                    value=m.get('joint_item_drawer_correct',m.get('preference_agreement'))
                    repetition=f"{m['largest_strategy_fraction']:.1%}" if task=='recovery' else '—'
                    rows.append(f"| {name} | {task} | {m['n_cases']} | {value:.1%} | {repetition} | {m['median_seconds']:.3f} / {m['p95_seconds']:.3f} |")
            display(Markdown('\n'.join(rows)))


## 4. Playground — a separate cell; no training

Run this cell, choose **Original saved run** or a **Retrained run**, and click **Load model**.

- **Your own interaction:** enter one item request and click **Compare replies**. Choose a reply to continue with a recovery/follow-up turn. Click **New interaction** for another item.
- **Replay original case:** select a case held out from the loaded fold and compare freshly computed replies against its recorded drawer or recovery preferences.
- **Save review:** records your judgment only. It does not change the model, routing, or original benchmark.

Replies use the actual saved SLM. Timing measures each new call, excluding model loading and warm-up. There is no response cache or simulated model fallback.
The SLM chooses the item/action/recovery strategy; wording uses grammatical templates. Interactive logs are saved separately from both original-data and Test N scores.


In [ ]:
try:
    import ipywidgets
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','ipywidgets==8.1.5'],check=True)
from google.colab import output
output.enable_custom_widget_manager()
if '_statelm_tester' in globals():_statelm_tester.stop()
import importlib.util
spec=importlib.util.spec_from_file_location('statelm_v41_notebook_ui',V41_PACKAGE_DIR/'statelm/notebook_ui.py')
ui=importlib.util.module_from_spec(spec);spec.loader.exec_module(ui)
new_runs=sorted([p for p in V41_RESULTS_ROOT.iterdir() if p.is_dir() and (p/'config.json').exists()],reverse=True)
runs=[('Retrained '+p.name,p) for p in new_runs]+[('Original saved run — 50.5% trained recovery',ORIGINAL_RUN_DIR)]
_statelm_tester=ui.show_playground(V41_PY,V41_PACKAGE_DIR,runs,V41_RESULTS_ROOT/'interactive_tests')


There should be a better way to chose a recovery route after failure. for that failure detection should be trained / obtained.

What model should look for:
- item,
- failure or not
- action

clarification could be different. like can you clarify, not which item you mean.

if item does not exist in the list it probably should say so.

in recovery I think better give the correct answer. we assume that model does not make a mistake twice.

we should also know how much flexible should be the inputs.

in the simulation, we should ahve an option of intentionally induce failure, without affecting its memory.


any of the models, including base slm does not distingush the failures properly

or should be some manual failure entry... idk.


In [ ]:
# StateLM: run THIS CELL after reconnecting. No earlier cells or training needed.
# Use a Colab GPU runtime and the Google Drive account containing your saved run.
from pathlib import Path
import base64, hashlib, importlib.util, io, json, os, shutil, subprocess, sys, zipfile
from google.colab import drive, output

SAVED_RUN = '20260913_110344_527137'
SEED, FOLD = 42, 4

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
_sim_drive = Path('/content/drive/MyDrive')
_sim_runs_root = _sim_drive / 'StateLM_v4_1_runs'
_sim_runs = sorted(
    [p for p in _sim_runs_root.glob('*') if p.is_dir() and (p / 'config.json').is_file()
     and any(p.glob('seed_*/fold_*/adapter/training_record.json'))], reverse=True)
if not _sim_runs:
    raise FileNotFoundError(
        'No saved v4.1 model found in MyDrive/StateLM_v4_1_runs. '
        'Mount the same Google Drive account used in the linked notebook.')
_sim_preferred = _sim_runs_root / SAVED_RUN
if _sim_preferred not in _sim_runs:
    print('The requested saved run is unavailable. Choose an available run below and click Load saved model.')
_sim_gpu = shutil.which('nvidia-smi')
if not _sim_gpu or subprocess.run([_sim_gpu, '-L'], capture_output=True).returncode:
    raise RuntimeError('Select Runtime → Change runtime type → GPU, then run this same cell again.')

# Release previous playground models when this cell is rerun in an existing runtime.
for _sim_name in ('_statelm_tester', '_statelm_original_tester', '_statelm_simulator'):
    _sim_old = globals().get(_sim_name)
    if _sim_old is not None:
        _sim_old.stop()
# An interrupted older loader could leave its worker alive before assigning it
# to the UI. Stop only instances of that previous simulation worker class.
_sim_previous_module = globals().get('_sim_ui')
if _sim_previous_module is not None and hasattr(_sim_previous_module, 'ModelProcess'):
    import gc
    for _sim_object in gc.get_objects():
        if isinstance(_sim_object, _sim_previous_module.ModelProcess):
            _sim_object.stop()
_sim_old_process = globals().get('_pg_process')
if _sim_old_process is not None and _sim_old_process.poll() is None:
    _sim_old_process.terminate()
    try:
        _sim_old_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        _sim_old_process.kill()
        _sim_old_process.wait(timeout=5)

# The small embedded package contains the linked notebook's existing inference
# code, reference inventory and data, plus the new two-tab interface. No weights.
_sim_bytes = base64.b64decode((
    '...'))
_sim_sha = '36864713fdb9c303cdf903e4e3155ea377630b775f415b7486c26f9f24839b87'
if hashlib.sha256(_sim_bytes).hexdigest() != _sim_sha:
    raise ValueError('The cell was not copied completely. Copy the whole cell again.')
_sim_package = Path('/content') / ('statelm_two_simulations_' + _sim_sha[:12])
_sim_package.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_sim_bytes)) as _sim_archive:
    for _sim_member in _sim_archive.infolist():
        _sim_target = (_sim_package / _sim_member.filename).resolve()
        if _sim_package.resolve() not in _sim_target.parents:
            raise ValueError('Invalid embedded package path')
    _sim_archive.extractall(_sim_package)
for _sim_name, _sim_expected in json.loads((_sim_package / 'MANIFEST.json').read_text()).items():
    if hashlib.sha256((_sim_package / _sim_name).read_bytes()).hexdigest() != _sim_expected:
        raise ValueError('Embedded file failed verification: ' + _sim_name)

# A separate, pinned runtime avoids changing Colab's already-imported libraries.
_sim_env = Path('/content/statelm_two_simulations_runtime_py312')
_sim_python = str(_sim_env / 'bin/python')
_sim_requirements = _sim_package / 'requirements-colab.txt'
_sim_dependencies_sha = hashlib.sha256(_sim_requirements.read_bytes()).hexdigest()
_sim_marker = _sim_env / '.dependencies_sha256'
if (not Path(_sim_python).is_file() or not _sim_marker.is_file()
        or _sim_marker.read_text() != _sim_dependencies_sha):
    print('Restoring the model environment. This can take several minutes after a runtime reset.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.6.17'], check=True)
    _sim_uv = shutil.which('uv')
    if not _sim_uv:
        raise RuntimeError('The environment installer was not found. Rerun this same cell.')
    subprocess.run([_sim_uv, 'python', 'install', '3.12.10'], check=True)
    if not Path(_sim_python).is_file():
        subprocess.run([_sim_uv, 'venv', '--python', '3.12.10', str(_sim_env)], check=True)
    subprocess.run([_sim_uv, 'pip', 'install', '--python', _sim_python,
                    '-r', str(_sim_requirements)], check=True)
    _sim_marker.write_text(_sim_dependencies_sha)
_sim_check = subprocess.run([_sim_python, '-c',
    "import torch; assert torch.cuda.is_available(), 'A working CUDA GPU is required'"],
    capture_output=True, text=True)
if _sim_check.returncode:
    raise RuntimeError('The GPU is unavailable. Reconnect to a Colab GPU and rerun this cell.\n' + _sim_check.stderr[-1000:])

os.environ['HF_HOME'] = '/content/statelm_v4_hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if not os.environ.get('HF_TOKEN'):
    from google.colab import userdata
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        from getpass import getpass
        os.environ['HF_TOKEN'] = getpass('Hugging Face token with Gemma access: ').strip()
if not os.environ.get('HF_TOKEN'):
    raise ValueError('Add your HF_TOKEN Colab secret with Gemma access, then rerun this cell.')
try:
    import ipywidgets
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets==8.1.5'], check=True)
output.enable_custom_widget_manager()
_sim_spec = importlib.util.spec_from_file_location('statelm_two_simulations_ui', _sim_package / 'simulation_ui.py')
_sim_ui = importlib.util.module_from_spec(_sim_spec)
_sim_spec.loader.exec_module(_sim_ui)
_statelm_simulator = _sim_ui.SimulatorUI(
    _sim_python, _sim_package, _sim_runs,
    _sim_drive / 'StateLM_v4_1_interactive_simulations', _sim_preferred, seed=SEED, fold=FOLD)
if _sim_preferred in _sim_runs:
    print('Loading the saved model. Both simulation sections are above; their buttons activate when Ready appears. No training is running.')
    _statelm_simulator.load()
